## Clase 10 - Ejercicio: regresión sobre el dataset Swiss

**INSERTE AQUÍ SU NOMBRE**

Para aprender sobre regresión, vamos a utilizar un dataset clásico llamado Swiss, que proviene originalmente del lenguaje R. Este dataset contiene datos socioeconómicos de 47 provincias suizas a fines del siglo XIX. Cada fila representa una provincia, y las variables reflejan características demográficas y sociales relevantes para ese contexto histórico.

## Variables

- `Location`: Provincia donde se midieron los datos.
- `Fertility`: Tasa de fertilidad (número promedio de hijos por mujer)
- `Agriculture`:` Porcentaje de hombres ocupados en agricultura
- `Examination`: Porcentaje de hombres que completaron exámenes de educación superior
- `Education`: Nivel promedio de educación (escala arbitraria)
- `Catholic`: Porcentaje de población católica
- `Infant.Mortality`: Tasa de mortalidad infantil (por cada 1000 nacidos vivos)

## Que queremos predecir?

Vamos a utilizar este dataset para predecir la tasa de fertilidad en cada provincia mediante diferentes métodos de regresión.

--- 

Siguiendo el procedimiento típico de Machine Learning, vamos a leer los datos y separarlos en los datasets de entrenamiento y testeo utilizando Scikit-Learn...

In [8]:
import pandas as pd

df = pd.read_csv("swiss.csv")

df.head()

,Location,Fertility,Agriculture,Examination,Education,Catholic,Infant.Mortality
0,Courtelary,80.2,17.0,15,12,9.96,22.2
1,Delemont,83.1,45.1,6,9,84.84,22.2
2,Franches-Mnt,92.5,39.7,5,5,93.40,20.2
3,Moutier,85.8,36.5,12,7,33.77,20.3
4,Neuveville,76.9,43.5,17,15,5.16,20.6


In [9]:
print(f"Tenemos {df.shape[0]} observaciones")

Tenemos 47 observaciones


Obtenemos la variable objetivo (`Fertility`) y, por otro lado, los atributos (quitamos `Location` ya que no es un atributo numérico relevante para la regresión)

In [10]:
X = df.drop(["Fertility", "Location"], axis=1)
y = df["Fertility"]

Dado que tenemos pocas observaciones, vamos a separar el dataset en un 50% para entrenamiento y 50% para testeo:

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

## Regresión lineal múltiple

Arranquemos la primera parte del ejercicio. Para eso, vamos a entrenar un modelo de regresión lineal múltiple usando todos los atributos. Para ello debes:

1. Escalar los atributos usando `StandardScaler`
2. Entrenar el modelo usando el dataset de entrenamiento.
3. Obtener las predicciones sobre el dataset de testeo.
4. Calcular las métricas MAE, MSE  y $R^2$, e imprimir los resultados.

In [12]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Regresión lineal múltiple con escalado
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LinearRegression())
])

# Entrenamiento
pipe_lr.fit(X_train, y_train)

# Predicciones sobre el set de testeo
y_pred_lr = pipe_lr.predict(X_test)

# Métricas
mae_lr = mean_absolute_error(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr  = r2_score(y_test, y_pred_lr)

print("=== Regresión Lineal Múltiple ===")
print(f"MAE: {mae_lr:.4f}")
print(f"MSE: {mse_lr:.4f}")
print(f"R²:  {r2_lr:.4f}")


=== Regresión Lineal Múltiple ===
MAE: 6.0937
MSE: 64.4572
R²:  0.5744


## Modelo con regularización

Para mejorar nuestro modelo, vamos a explorar técnicas de regresión lineal con regularización, que nos permiten controlar el sobreajuste y seleccionar variables relevantes automáticamente.

Existen dos variantes muy populares:

- Una penaliza la suma de los cuadrados de los coeficientes (regularización L2).
- La otra penaliza la suma del valor absoluto de los coeficientes (regularización L1).

Ambas ayudan a mejorar la generalización, pero una de ellas además puede eliminar variables (coeficientes exactamente cero), lo que ayuda a identificar qué atributos son realmente importantes.

Tu tarea:

1. Elegí correctamente cuál de los dos métodos de regularización usar para este problema. 
    - Pista: Queremos que el modelo sea capaz de hacer una selección automática de variables, dejando fuera aquellas que no aportan.
2. Implementá un pipeline que incluya escalado y el modelo elegido.
3. Buscá automáticamente el mejor valor del hiperparámetro de regularización (alpha) usando validación cruzada usando 3-folds.
4. Entrená el modelo con los datos de entrenamiento y obtené las predicciones para el set de testeo.
5. Calcular las métricas MAE, MSE  y $R^2$, e imprimir los resultados.
6. Imprimí los coeficientes resultantes e identificá qué variables fueron eliminadas (coeficiente = 0).

In [13]:
import numpy as np

from sklearn.linear_model import LassoCV, RidgeCV

Paremos un momento para entender qué hacen LassoCV y RidgeCV antes de continuar con la resolución:

> Tanto `LassoCV` como `RidgeCV` son implementaciones de regresión lineal con regularización que incluyen la búsqueda automática del mejor hiperparámetro alpha mediante validación cruzada.
>
> Ambos métodos prueban distintos valores de alpha y eligen el que minimiza el error del modelo, facilitando el proceso de ajuste sin necesidad de una búsqueda manual.
>
> Internamente, utilizan la métrica del error cuadrático medio (MSE) para evaluar el rendimiento del modelo en cada fold de la validación cruzada.
>
> Por ejemplo, si llamás a RidgeCV(alphas=alphas, cv=5), se hará una validación cruzada de 5 folds utilizando los valores de alpha que vos le pases, y se seleccionará el que obtenga el menor MSE promedio.
> 
> Una vez elegido el mejor alpha, el modelo final se entrena con todos los datos de entrenamiento usando ese valor.

¡Listo! Con todo lo que vimos hasta ahora, ya estás en condiciones de resolver esta parte y completar los 6 puntos propuestos

In [14]:
alphas = np.logspace(-4, 1, 500)

# Modelo regularizado con Lasso (selección automática de variables)
# Usamos LassoCV para buscar el mejor alpha con validación cruzada (cv=3)
pipe_lasso = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", LassoCV(alphas=alphas, cv=3, max_iter=10000, random_state=42))
])

# Entrenamiento (LassoCV elige automáticamente el mejor alpha)
pipe_lasso.fit(X_train, y_train)

# Mejor alpha encontrado
best_alpha = pipe_lasso.named_steps["lasso"].alpha_
print(f"Mejor alpha encontrado con CV (3 folds): {best_alpha:.6f}\n")

# Predicciones sobre el set de testeo
y_pred_lasso = pipe_lasso.predict(X_test)

# Métricas
mae_lasso = mean_absolute_error(y_test, y_pred_lasso)
mse_lasso = mean_squared_error(y_test, y_pred_lasso)
r2_lasso  = r2_score(y_test, y_pred_lasso)

print("=== Lasso (Regularización L1) ===")
print(f"MAE: {mae_lasso:.4f}")
print(f"MSE: {mse_lasso:.4f}")
print(f"R²:  {r2_lasso:.4f}\n")

# Coeficientes e identificación de variables eliminadas
coef_lasso = pipe_lasso.named_steps["lasso"].coef_
print("Coeficientes por variable:")
for var, coef in zip(X.columns, coef_lasso):
    if abs(coef) < 1e-10:
        print(f"  - {var}: {coef:.6f}  -> ELIMINADA (coeficiente = 0)")
    else:
        print(f"  - {var}: {coef:.6f}")


Mejor alpha encontrado con CV (3 folds): 2.181109

=== Lasso (Regularización L1) ===
MAE: 5.9954
MSE: 64.2943
R²:  0.5755

Coeficientes por variable:
  - Agriculture: -0.000000  -> ELIMINADA (coeficiente = 0)
  - Examination: -1.762433
  - Education: -2.404905
  - Catholic: 1.719649
  - Infant.Mortality: 3.200407


## Comparación de modelos y conclusiones

Completá la siguiente tabla con las métricas obtenidas para cada uno de los modelos que entrenaste:

| Modelo                        | MAE | MSE | $R^2$ |
| ----------------------------- | --- | --- | ----- |
| Regresión Lineal              |     |     |       |
| Modelo Regularizado (L1 o L2) |     |     |       |


> ⚠️ Asegurate de cambiar el nombre del modelo `Modelo Regularizado (L1 o L2)` según el modelo que usaste (Lasso o Ridge).

### Justificación

**¿Cuál de los modelos te parece que tuvo un mejor desempeño general?**

Tené en cuenta las tres métricas al responder, y también pensá en la complejidad del modelo (por ejemplo, si eliminó variables innecesarias).

Escribí tu respuesta a continuación:

| Modelo                        | MAE    | MSE      | $R^2$  |
| ----------------------------- | ------ | -------- | ------ |
| Regresión Lineal              | 6.0937 | 64.4572  | 0.5744 |
| Lasso (Regularización L1)     | 5.9954 | 64.2943  | 0.5755 |

**Justificación:**

Con los valores obtenidos, **Lasso** presenta un desempeño ligeramente mejor que la regresión lineal múltiple en las tres métricas (menor MAE y MSE, y mayor $R^2$). Además, gracias a la penalización L1, Lasso realizó selección automática de variables: **Agriculture** quedó eliminada ($coeficiente = 0$), mientras que el resto de las variables (`Examination`, `Education`, `Catholic`, `Infant.Mortality`) se mantuvieron con coeficientes distintos de cero. 

Teniendo en cuenta tanto las métricas como la complejidad del modelo, **Lasso resulta más adecuado para este problema**: logra una generalización comparable (ligeramente superior) con un modelo más parsimonioso, al descartar una variable que no aporta al predecir la tasa de fertilidad. Esto reduce el riesgo de sobreajuste y mejora la interpretabilidad del modelo.
